# 03 Parse PDFs with Docling

## Σκοπός
Σε αυτό το notebook:

- φορτώνουμε το working FinanceBench dataset
- εντοπίζουμε τα αντίστοιχα local PDFs
- κάνουμε parse τα PDFs με Docling
- αποθηκεύουμε markdown και structured JSON
- δημιουργούμε manifest με parse status
- εξάγουμε zip αρχείο με όλα τα markdown outputs

Το notebook έχει σχεδιαστεί ώστε να μπορεί να τρέξει τόσο τοπικά όσο και σε Kaggle.

In [ ]:
import sys

IN_KAGGLE = "kaggle_secrets" in sys.modules or "/kaggle/working" in str(__import__("pathlib").Path.cwd())

print("IN_KAGGLE:", IN_KAGGLE)

In [ ]:
# Uncomment if needed
# !pip install -q docling pandas pyarrow tqdm

In [ ]:
from pathlib import Path
import json
import warnings
import zipfile
import shutil
from datetime import datetime

import pandas as pd
from tqdm.auto import tqdm

from docling.document_converter import DocumentConverter

In [ ]:
warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 200)
pd.set_option("display.width", 140)

IN_KAGGLE = Path("/kaggle/working").exists()
print("IN_KAGGLE:", IN_KAGGLE)

In [ ]:
CURRENT_DIR = Path.cwd()

if CURRENT_DIR.name == "notebooks":
    BASE_DIR = CURRENT_DIR.parent
else:
    BASE_DIR = CURRENT_DIR

DATA_DIR = BASE_DIR / "data"
RAW_DIR = DATA_DIR / "raw"
PDFS_DIR = RAW_DIR / "pdfs"

INTERIM_DIR = DATA_DIR / "interim"
MARKDOWN_DIR = INTERIM_DIR / "markdown"
PARSED_JSON_DIR = INTERIM_DIR / "parsed_json"

OUTPUTS_DIR = BASE_DIR / "outputs"
LOGS_DIR = OUTPUTS_DIR / "logs"

MANIFEST_PATH = INTERIM_DIR / "docling_parse_manifest.csv"
ZIP_OUTPUT_PATH = OUTPUTS_DIR / "financebench_markdown.zip"

for p in [MARKDOWN_DIR, PARSED_JSON_DIR, OUTPUTS_DIR, LOGS_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print("BASE_DIR:", BASE_DIR)
print("MARKDOWN_DIR:", MARKDOWN_DIR)
print("PARSED_JSON_DIR:", PARSED_JSON_DIR)
print("OUTPUTS_DIR:", OUTPUTS_DIR)

In [ ]:
# Optional Kaggle override
# Example:
# BASE_DIR = Path("/kaggle/working")
# DATA_DIR = BASE_DIR / "data"
# RAW_DIR = DATA_DIR / "raw"
# PDFS_DIR = RAW_DIR / "pdfs"
# INTERIM_DIR = DATA_DIR / "interim"
# MARKDOWN_DIR = INTERIM_DIR / "markdown"
# PARSED_JSON_DIR = INTERIM_DIR / "parsed_json"
# OUTPUTS_DIR = BASE_DIR / "outputs"
# WORKING_DATASET_PATH = INTERIM_DIR / "financebench_open_source_working.csv"

In [ ]:
candidate_paths = [
    INTERIM_DIR / "financebench_open_source_working.csv",
    Path("/kaggle/input/datasets/theofanisnikolaou/financebench-data/financebench_open_source_working.csv"),
]

if Path("/kaggle/input").exists():
    candidate_paths.extend(Path("/kaggle/input").glob("**/financebench_open_source_working.csv"))

existing_paths = [p for p in candidate_paths if p.exists()]

print("Candidate paths checked:")
for p in candidate_paths:
    print("-", p)

if not existing_paths:
    raise FileNotFoundError("financebench_open_source_working.csv was not found.")

WORKING_DATASET_PATH = existing_paths[0]
print("\nUsing WORKING_DATASET_PATH:", WORKING_DATASET_PATH)

In [ ]:
df = pd.read_csv(WORKING_DATASET_PATH)

print("Loaded shape:", df.shape)
df.head(2)

In [ ]:
KAGGLE_PDFS_DIR = Path("/kaggle/input/datasets/theofanisnikolaou/financebench-data/pdfs")

if IN_KAGGLE:
    df["pdf_path"] = df["pdf_filename"].apply(
        lambda x: str(KAGGLE_PDFS_DIR / x) if pd.notna(x) else None
    )

print("KAGGLE_PDFS_DIR exists:", KAGGLE_PDFS_DIR.exists())
df[["doc_name", "pdf_filename", "pdf_path"]].head()

In [ ]:
required_cols = ["doc_name", "pdf_filename", "pdf_path"]

missing_cols = [c for c in required_cols if c not in df.columns]
if missing_cols:
    raise ValueError(f"Missing required columns: {missing_cols}")

print("All required columns exist.")

In [ ]:
def safe_stem(s: str) -> str:
    s = str(s).strip()
    for ch in ["/", "\\", ":", "*", "?", '"', "<", ">", "|"]:
        s = s.replace(ch, "_")
    return s

In [ ]:
base_cols = ["doc_name", "pdf_filename", "pdf_path"]
optional_cols = ["company", "doc_type", "doc_period"]

selected_cols = base_cols + [c for c in optional_cols if c in df.columns]

docs_df = (
    df[selected_cols]
    .drop_duplicates()
    .reset_index(drop=True)
)

docs_df["output_stem"] = docs_df["doc_name"].apply(safe_stem)
docs_df["pdf_exists"] = docs_df["pdf_path"].apply(lambda x: Path(x).exists() if pd.notna(x) else False)
docs_df["pdf_size_bytes"] = docs_df["pdf_path"].apply(
    lambda x: Path(x).stat().st_size if pd.notna(x) and Path(x).exists() else None
)

print("Unique documents:", len(docs_df))
docs_df.head()

In [ ]:
docs_df["pdf_exists"].value_counts(dropna=False).to_frame("count")

In [ ]:
missing_pdf_df = docs_df[~docs_df["pdf_exists"]].copy()
print("Missing PDF files:", len(missing_pdf_df))
missing_pdf_df.head(10)

In [ ]:
converter = DocumentConverter()
print("Docling converter initialized.")

In [ ]:
def save_text(text: str, path: Path):
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(text, encoding="utf-8")

def save_json(data, path: Path):
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)

def now_iso():
    return datetime.utcnow().isoformat()

In [ ]:
def parse_pdf_with_docling(pdf_path: Path, markdown_path: Path, json_path: Path):
    result = converter.convert(str(pdf_path))
    doc = result.document

    markdown_text = doc.export_to_markdown()
    save_text(markdown_text, markdown_path)

    try:
        doc_dict = doc.model_dump()
    except AttributeError:
        try:
            doc_dict = doc.dict()
        except Exception:
            doc_dict = {"warning": "Could not serialize document with model_dump/dict"}

    save_json(doc_dict, json_path)

    return {
        "markdown_chars": len(markdown_text),
        "json_saved": True,
        "markdown_saved": True
    }

In [ ]:
records = []

parse_input_df = docs_df[docs_df["pdf_exists"]].copy().reset_index(drop=True)

print("Documents eligible for parsing:", len(parse_input_df))

if parse_input_df.empty:
    print("No documents to parse. Check pdf_path and pdf_exists.")
    manifest_df = pd.DataFrame(columns=[
        "doc_name",
        "company",
        "doc_type",
        "doc_period",
        "pdf_filename",
        "pdf_path",
        "markdown_path",
        "json_path",
        "parse_started_at",
        "parse_finished_at",
        "status",
        "error_message",
        "markdown_chars",
        "json_saved",
        "markdown_saved"
    ])
else:
    for _, row in tqdm(parse_input_df.iterrows(), total=len(parse_input_df), desc="Parsing PDFs"):
        doc_name = row["doc_name"]
        pdf_filename = row["pdf_filename"]
        pdf_path = Path(row["pdf_path"])
        output_stem = row.get("output_stem", safe_stem(row["doc_name"]))

        markdown_path = MARKDOWN_DIR / f"{output_stem}.md"
        json_path = PARSED_JSON_DIR / f"{output_stem}.json"

        record = {
            "doc_name": doc_name,
            "company": row.get("company"),
            "doc_type": row.get("doc_type"),
            "doc_period": row.get("doc_period"),
            "pdf_filename": pdf_filename,
            "pdf_path": str(pdf_path),
            "markdown_path": str(markdown_path),
            "json_path": str(json_path),
            "parse_started_at": now_iso(),
            "status": None,
            "error_message": None,
            "markdown_chars": None,
            "json_saved": False,
            "markdown_saved": False
        }

        try:
            parse_info = parse_pdf_with_docling(
                pdf_path=pdf_path,
                markdown_path=markdown_path,
                json_path=json_path
            )
            record.update(parse_info)
            record["status"] = "success"

        except Exception as e:
            record["status"] = "error"
            record["error_message"] = str(e)

        record["parse_finished_at"] = now_iso()
        records.append(record)

    manifest_df = pd.DataFrame(records)

print("Parsing finished.")
print("manifest_df shape:", manifest_df.shape)
manifest_df.head()

In [ ]:
manifest_df.to_csv(MANIFEST_PATH, index=False)
print("Manifest saved to:", MANIFEST_PATH)

In [ ]:
if manifest_df.empty:
    summary = {
        "total_docs_attempted": 0,
        "success_count": 0,
        "error_count": 0,
        "markdown_files_created": 0,
        "json_files_created": 0
    }
else:
    summary = {
        "total_docs_attempted": len(manifest_df),
        "success_count": int((manifest_df["status"] == "success").sum()),
        "error_count": int((manifest_df["status"] == "error").sum()),
        "markdown_files_created": int(manifest_df["markdown_saved"].fillna(False).sum()),
        "json_files_created": int(manifest_df["json_saved"].fillna(False).sum())
    }

pd.DataFrame([summary])

In [ ]:
if manifest_df.empty:
    print("Manifest is empty.")
else:
    errors_df = manifest_df[manifest_df["status"] == "error"].copy()
    print("Errors:", len(errors_df))
    errors_df[["doc_name", "pdf_filename", "error_message"]].head(20)

In [ ]:
md_files = sorted(MARKDOWN_DIR.glob("*.md"))

print("Markdown files found:", len(md_files))
for f in md_files[:10]:
    print("-", f.name)

In [ ]:
if md_files:
    sample_md_path = md_files[0]
    sample_md_text = sample_md_path.read_text(encoding="utf-8")
    print("Sample file:", sample_md_path.name)
    print(sample_md_text[:3000])
else:
    print("No markdown files found.")

In [ ]:
with zipfile.ZipFile(ZIP_OUTPUT_PATH, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    # markdown files
    for md_file in MARKDOWN_DIR.glob("*.md"):
        zf.write(md_file, arcname=f"markdown/{md_file.name}")

    # json files
    for json_file in PARSED_JSON_DIR.glob("*.json"):
        zf.write(json_file, arcname=f"parsed_json/{json_file.name}")

    # manifest
    if MANIFEST_PATH.exists():
        zf.write(MANIFEST_PATH, arcname=f"manifest/{MANIFEST_PATH.name}")

print("ZIP created:", ZIP_OUTPUT_PATH)
print("ZIP size (bytes):", ZIP_OUTPUT_PATH.stat().st_size if ZIP_OUTPUT_PATH.exists() else None)

In [ ]:
if IN_KAGGLE:
    kaggle_zip_path = Path("/kaggle/working/financebench_markdown.zip")
    if ZIP_OUTPUT_PATH.resolve() != kaggle_zip_path.resolve():
        shutil.copy2(ZIP_OUTPUT_PATH, kaggle_zip_path)
    print("Kaggle ZIP path:", kaggle_zip_path)
else:
    print("Running outside Kaggle.")

In [ ]:
output_summary = pd.DataFrame([{
    "manifest_path": str(MANIFEST_PATH),
    "markdown_dir": str(MARKDOWN_DIR),
    "parsed_json_dir": str(PARSED_JSON_DIR),
    "zip_output_path": str(ZIP_OUTPUT_PATH),
    "n_markdown_files": len(list(MARKDOWN_DIR.glob("*.md"))),
    "n_json_files": len(list(PARSED_JSON_DIR.glob("*.json")))
}])

output_summary

## Συμπέρασμα

Σε αυτό το notebook:

- κάναμε parse τα διαθέσιμα local PDFs με Docling
- αποθηκεύσαμε markdown και structured JSON
- δημιουργήσαμε manifest με parse status ανά document
- δημιουργήσαμε zip αρχείο με όλα τα markdown outputs

Το επόμενο notebook θα είναι το `04_clean_markdown_and_inspect.ipynb`.